# Smart Demand Signals: Forecasting, Client Classification, PCA, and Validation

This notebook implements the same analytical logic as `project.py`, but in an exploratory and explainable format for the hackathon presentation.

The goal is to transform transaction history into operational commercial alerts:

- Aggregate sales weekly.
- Forecast the next 4 weeks of product-type and product-level demand.
- Classify every client/category into `Loyal`, `Promiscuous`, `Risky`, or `Promising`.
- Predict future client state and raise prioritized alarms.
- Explain each alarm using the most influential product driver.
- Validate the future-state inference with rolling 5-fold time-series cross-validation.
- Use standardization where it is mathematically useful: PCA and logistic regression.

The production pipeline remains in `project.py`; this notebook imports those functions so the script and notebook stay consistent.

## 0. Environment Setup

If the next import cell fails, install the dependencies first:

```bash
python -m pip install -r requirements.txt
```

The notebook intentionally does not auto-install packages every run, because hackathon/demo machines can have different Python environments.

In [ ]:
import importlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown, IFrame

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

import project as sd
importlib.reload(sd)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Parameters

The default forecast horizon is 4 weeks. The model mode can be:

- `auto`: use SARIMA/SARIMAX when `statsmodels` can fit the model; otherwise fallback automatically.
- `fallback`: use deterministic seasonal/rolling forecasts for faster debugging.

For the final run, `auto` is recommended. For quick CV debugging, you can switch `CV_MODEL_MODE` to `fallback`.

The state engine is intentionally business-configurable. Change `CLASSIFICATION_CONFIG` to decide the potential-achieved threshold for stable clients and the slope threshold for growth/decline clients.

SARIMA/SARIMAX orders are also configured here. The notebook uses `MODEL_CONFIG["type_order"]` and `MODEL_CONFIG["type_seasonal_order"]` for the analytical-block SARIMA models, and `MODEL_CONFIG["product_order"]` / `MODEL_CONFIG["product_seasonal_order"]` for product-level SARIMAX models.

### General Run Variables

- `INPUT_FILE`: Excel workbook containing `Ventas`, `Productos`, `Clientes`, `Potencial`, and `Campañas`.
- `OUTPUT_DIR`: folder where the notebook writes CSV, Excel, HTML PCA plots, and validation outputs.
- `FORECAST_WEEKS`: number of future weeks to forecast. The hackathon design uses `4`.
- `MODEL_MODE`: `auto` fits SARIMA/SARIMAX when possible; `fallback` skips statistical fitting and uses a seasonal/rolling baseline.
- `CV_MODEL_MODE`: model mode used inside rolling cross-validation. Set to `fallback` if validation is too slow.
- `MAXITER`: maximum optimizer iterations for each SARIMA/SARIMAX fit. Higher can improve convergence but slows execution.
- `TOP_ALERTS`: number of top-priority alerts saved in the compact alert output.

### SARIMA/SARIMAX Variables

- `type_order`: non-seasonal SARIMA `(p, d, q)` for total analytical-block demand. `p` is autoregressive order, `d` is differencing, and `q` is moving-average order.
- `type_seasonal_order`: seasonal SARIMA `(P, D, Q, s)` for total analytical-block demand. `s=52` means yearly seasonality in weekly data.
- `product_order`: non-seasonal SARIMAX `(p, d, q)` for each product time series.
- `product_seasonal_order`: seasonal SARIMAX `(P, D, Q, s)` for each product. The default is no product-level seasonality to avoid overfitting sparse SKUs.
- `min_model_weeks`: minimum weekly observations required before attempting SARIMA/SARIMAX. Shorter series use fallback.
- `min_seasonal_weeks`: minimum weekly observations required before enabling seasonal terms. With `104`, the model needs about two years of weekly history.

### Classification Threshold Variables

Each analytical block has its own threshold dictionary inside `CLASSIFICATION_CONFIG`.

- `potential_achieved_threshold`: if the slope is neutral and `spend_4w / potential_4w` is greater than or equal to this value, the client is `Loyal`; otherwise the client is `Promiscuous`.
- `slope_threshold`: if slope is greater than or equal to this value, the client is `Promising`; if slope is less than or equal to the negative of this value, the client is `Risky`.
- `default`: fallback thresholds used if a product block appears that is not explicitly listed.

Classification priority is: slope first, potential second. That means `Loyal` and `Promiscuous` are only assigned when the slope is between `-slope_threshold` and `+slope_threshold`.

### State Order

- `STATE_ORDER`: display and matrix order for classification reports, confusion matrices, and transition matrices.

In [ ]:
INPUT_FILE = Path("Datasets.xlsx")
OUTPUT_DIR = Path("outputs_notebook")

FORECAST_WEEKS = 4
MODEL_MODE = "auto"       # "auto" or "fallback"
CV_MODEL_MODE = MODEL_MODE # set to "fallback" if CV is too slow on your laptop
MAXITER = 60
TOP_ALERTS = 250

MODEL_CONFIG = {
    "type_order": (1, 1, 1),
    "type_seasonal_order": (1, 0, 0, 52),
    "product_order": (1, 0, 1),
    "product_seasonal_order": (0, 0, 0, 0),
    "min_model_weeks": 36,
    "min_seasonal_weeks": 104,
}

CLASSIFICATION_CONFIG = {
    "Commodities": {
        "potential_achieved_threshold": 0.60,
        "slope_threshold": 0.15,
    },
    "Productos Técnicos": {
        "potential_achieved_threshold": 0.45,
        "slope_threshold": 0.30,
    },
    "default": {
        "potential_achieved_threshold": 0.55,
        "slope_threshold": 0.20,
    },
}

STATE_ORDER = ["Loyal", "Promising", "Risky", "Promiscuous"]

OUTPUT_DIR.mkdir(exist_ok=True)
display(Markdown(f"**Input file:** `{INPUT_FILE}`  \n**Output directory:** `{OUTPUT_DIR}`"))
display(pd.DataFrame(CLASSIFICATION_CONFIG).T)
display(Markdown("### SARIMA/SARIMAX Configuration"))
display(pd.DataFrame([MODEL_CONFIG]))

## 2. Load, Clean, and Join Data

The workbook contains five business tables:

- `Ventas`: transactional sales at SKU level.
- `Productos`: product hierarchy and analytical block.
- `Clientes`: client geography.
- `Potencial`: estimated purchase potential by client/category.
- `Campañas`: promotional campaign calendar.

The cleaning logic normalizes IDs, converts Excel serial dates, keeps returns/credits as negative revenue, and joins product/client context onto sales.

In [ ]:
sheets = sd.load_workbook(INPUT_FILE)
data = sd.prepare_data(sheets)

sales = data["sales"]
products = data["products"]
clients = data["clients"]
potential = data["potential"]
campaigns = data["campaigns"]

summary = pd.DataFrame({
    "table": ["sales", "products", "clients", "potential", "campaigns"],
    "rows": [len(sales), len(products), len(clients), len(potential), len(campaigns)],
    "columns": [sales.shape[1], products.shape[1], clients.shape[1], potential.shape[1], campaigns.shape[1]],
})
display(summary)

display(Markdown(
    f"Sales cover **{sales['date'].min().date()}** to **{sales['date'].max().date()}**, "
    f"with **{sales['client_id'].nunique():,}** clients and **{sales['product_id'].nunique():,}** products."
))

## 3. Weekly Aggregation

Weekly aggregation is the base grain of the project. It smooths daily noise while staying actionable for a sales team.

Client classification uses a rolling 4-week window, and reliability uses a 12-week context window.

In [ ]:
current_week, history_weeks, future_weeks, all_weeks = sd.build_week_calendar(sales, current_end=None, forecast_weeks=FORECAST_WEEKS)
campaign_calendar = sd.build_campaign_calendar(campaigns, all_weeks)
weekly_sales = sd.build_weekly_sales(sales, campaign_calendar)
weekly_sales = weekly_sales[weekly_sales["week_start"].isin(history_weeks)].copy()

display(Markdown(
    f"**Current analysis week:** `{current_week.date()}`  \n"
    f"**Forecast weeks:** {', '.join(str(d.date()) for d in future_weeks)}  \n"
    f"**Weekly sales rows:** {len(weekly_sales):,}"
))

weekly_block = weekly_sales.groupby(["week_start", "analytical_block"], as_index=False)["revenue"].sum()
plt.figure(figsize=(13, 5))
sns.lineplot(data=weekly_block, x="week_start", y="revenue", hue="analytical_block")
plt.title("Weekly Revenue by Analytical Block")
plt.xlabel("Week")
plt.ylabel("Net revenue")
plt.tight_layout()
plt.show()

## 4. Hierarchical Forecasting

The forecasting layer is hierarchical:

1. Forecast weekly revenue by product type/block: `Commodities` and `Productos Técnicos`.
2. Convert type forecasts into demand factors.
3. Forecast each product using SARIMAX with the type demand factor as an exogenous signal.
4. Use product forecasts as explanatory signals in client-level future classification.

This keeps the model commercially interpretable and avoids fitting fragile client/product time series.

In [ ]:
type_forecasts, factor_by_block = sd.forecast_type_sales(
    weekly_sales, history_weeks, future_weeks, MODEL_MODE, MAXITER, MODEL_CONFIG
)
product_forecasts = sd.forecast_product_sales(
    weekly_sales, products, history_weeks, future_weeks, factor_by_block, MODEL_MODE, MAXITER, MODEL_CONFIG
)

display(type_forecasts)
display(product_forecasts.head())

model_use = product_forecasts.groupby(["model_method", "analytical_block"], as_index=False)["product_id"].nunique()
display(Markdown("### Product Forecast Model Usage"))
display(model_use)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=type_forecasts, x="analytical_block", y="forecast_revenue", hue="week_start", ax=axes[0])
axes[0].set_title("Next 4 Weeks Forecast by Product Type")
axes[0].tick_params(axis="x", rotation=20)

top_products = product_forecasts.groupby("product_id", as_index=False)["forecast_4w_revenue"].first().nlargest(10, "forecast_4w_revenue")
sns.barplot(data=top_products, y="product_id", x="forecast_4w_revenue", ax=axes[1], orient="h")
axes[1].set_title("Top Forecasted Products, Next 4 Weeks")
axes[1].set_xlabel("Forecast revenue")
axes[1].set_ylabel("Product")
plt.tight_layout()
plt.show()

display(Markdown("### Analytical-Block SARIMA Forecasts: Actual vs Next 4 Weeks"))
block_history = weekly_sales.groupby(["week_start", "analytical_block"], as_index=False)["revenue"].sum()
for block in sorted(block_history["analytical_block"].dropna().unique()):
    hist = block_history[block_history["analytical_block"] == block].tail(104).copy()
    fcst = type_forecasts[type_forecasts["analytical_block"] == block].copy()
    plt.figure(figsize=(13, 4.5))
    sns.lineplot(data=hist, x="week_start", y="revenue", label="Historical weekly revenue")
    sns.lineplot(data=fcst, x="week_start", y="forecast_revenue", marker="o", label="SARIMA forecast")
    if not hist.empty:
        plt.axvline(hist["week_start"].max(), color="gray", linestyle="--", alpha=0.7, label="Forecast start")
    plt.title(f"{block}: SARIMA Actual vs Forecast")
    plt.xlabel("Week")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()

display(Markdown("### Product SARIMAX Forecasts: Top Products by Forecasted Revenue"))
product_history = weekly_sales.groupby(["week_start", "product_id", "analytical_block"], as_index=False)["revenue"].sum()
top_product_ids = (
    product_forecasts.groupby("product_id", as_index=False)["forecast_4w_revenue"]
    .first()
    .nlargest(8, "forecast_4w_revenue")["product_id"]
    .tolist()
)
for product_id in top_product_ids:
    meta = product_forecasts[product_forecasts["product_id"] == product_id].iloc[0]
    hist = product_history[product_history["product_id"] == product_id].tail(104).copy()
    fcst = product_forecasts[product_forecasts["product_id"] == product_id].copy()
    plt.figure(figsize=(13, 4.5))
    sns.lineplot(data=hist, x="week_start", y="revenue", label="Historical weekly revenue")
    sns.lineplot(data=fcst, x="week_start", y="forecast_revenue", marker="o", label="SARIMAX forecast")
    if not hist.empty:
        plt.axvline(hist["week_start"].max(), color="gray", linestyle="--", alpha=0.7, label="Forecast start")
    plt.title(f"Product {product_id} ({meta['analytical_block']}): SARIMAX Actual vs Forecast")
    plt.xlabel("Week")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()

## 5. Reference SARIMA/SARIMAX Fit Inspection

The pipeline stores forecasts and fallback status. For interpretation, this section fits one representative type model and one representative product model so we can inspect estimated parameters directly.

If a model fails or falls back, that is not automatically bad: sparse series often should not be forced into a complex SARIMAX model.

In [ ]:
def fit_reference_models(weekly_sales, products, history_weeks, factor_by_block):
    block_weekly = weekly_sales.groupby(["analytical_block", "week_start"], as_index=False)["revenue"].sum()
    best_block = block_weekly.groupby("analytical_block")["revenue"].sum().idxmax()
    y_type = block_weekly[block_weekly["analytical_block"] == best_block].set_index("week_start")["revenue"].reindex(history_weeks, fill_value=0.0)
    type_model = SARIMAX(y_type, order=MODEL_CONFIG["type_order"], seasonal_order=MODEL_CONFIG["type_seasonal_order"], enforce_stationarity=False, enforce_invertibility=False)
    type_fit = type_model.fit(disp=False, maxiter=MAXITER)

    product_totals = weekly_sales.groupby("product_id", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
    best_product = product_totals.iloc[0]["product_id"]
    meta = products.set_index("product_id").loc[best_product]
    product_weekly = weekly_sales.groupby(["product_id", "week_start"], as_index=False)["revenue"].sum()
    y_product = product_weekly[product_weekly["product_id"] == best_product].set_index("week_start")["revenue"].reindex(history_weeks, fill_value=0.0)
    block_factor = factor_by_block.get(meta["analytical_block"], pd.Series(1.0, index=history_weeks))
    exog = pd.DataFrame({"type_demand_factor": block_factor.reindex(history_weeks).fillna(1.0)}, index=history_weeks)
    product_model = SARIMAX(y_product, exog=exog, order=MODEL_CONFIG["product_order"], seasonal_order=MODEL_CONFIG["product_seasonal_order"], enforce_stationarity=False, enforce_invertibility=False)
    product_fit = product_model.fit(disp=False, maxiter=MAXITER)
    return best_block, type_fit, best_product, meta, product_fit

try:
    best_block, type_fit, best_product, best_product_meta, product_fit = fit_reference_models(weekly_sales, products, history_weeks, factor_by_block)
    display(Markdown(f"### Reference Type Model: `{best_block}`"))
    display(type_fit.summary())
    type_ref = pd.DataFrame({
        "week_start": type_fit.data.row_labels,
        "actual": np.asarray(type_fit.model.endog).ravel(),
        "fitted": np.asarray(type_fit.fittedvalues).ravel(),
    }).tail(104)
    plt.figure(figsize=(13, 4.5))
    sns.lineplot(data=type_ref, x="week_start", y="actual", label="Actual")
    sns.lineplot(data=type_ref, x="week_start", y="fitted", label="Fitted")
    plt.title(f"Reference Type SARIMA Fitted Values: {best_block}")
    plt.xlabel("Week")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()

    display(Markdown(f"### Reference Product SARIMAX: product `{best_product}` in `{best_product_meta['analytical_block']}`"))
    display(product_fit.summary())
    product_ref = pd.DataFrame({
        "week_start": product_fit.data.row_labels,
        "actual": np.asarray(product_fit.model.endog).ravel(),
        "fitted": np.asarray(product_fit.fittedvalues).ravel(),
    }).tail(104)
    plt.figure(figsize=(13, 4.5))
    sns.lineplot(data=product_ref, x="week_start", y="actual", label="Actual")
    sns.lineplot(data=product_ref, x="week_start", y="fitted", label="Fitted")
    plt.title(f"Reference Product SARIMAX Fitted Values: Product {best_product}")
    plt.xlabel("Week")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    display(Markdown(f"Reference model inspection could not be fitted: `{exc}`"))

## 6. Client Feature Engineering and Configurable Classification

Each client/category receives current and forecasted features:

- `spend_4w`, `prev_spend_4w`, `spend_12w`
- `capture_ratio`: spend versus 4-week potential
- `predicted_spend_4w`: recent behavior adjusted by client trend and product forecasts
- `classification_observed_slope`: client-weighted average historical slope of the products the client buys most.
- `classification_forecast_slope`: client-weighted average forecasted slope of the products the client buys most, derived from product SARIMAX forecasts.
- `current_state` and `future_state`

Classification now has only four final statuses:

- `Promising`: slope is greater than or equal to `slope_threshold`.
- `Risky`: slope is less than or equal to `-slope_threshold`.
- `Loyal`: slope is neutral and capture ratio is greater than or equal to `potential_achieved_threshold`.
- `Promiscuous`: slope is neutral and capture ratio is below `potential_achieved_threshold`.

The thresholds can differ for commodities and technical products.

In [ ]:
classifications, product_driver = sd.build_client_features(
    weekly_sales, potential, current_week, history_weeks, product_forecasts, CLASSIFICATION_CONFIG
)
alerts = sd.build_alerts(classifications, product_driver)
transitions = sd.build_transition_tables(classifications)

display(classifications.head())
display(alerts.head(20))

eligible_classifications = classifications[classifications["classification_eligible"]].copy()
eligible_state_summary = (
    eligible_classifications.groupby(["analytical_block", "current_state"], as_index=False)
    .size()
    .assign(pct=lambda d: d["size"] / d.groupby("analytical_block")["size"].transform("sum"))
)
display(Markdown("### Current State Distribution"))
display(eligible_state_summary)
display(Markdown("### Transition Probabilities"))
display(transitions["transition_probabilities"])

state_counts = classifications.melt(
    id_vars=["client_id", "category_h", "analytical_block"],
    value_vars=["current_state", "future_state"],
    var_name="period",
    value_name="state",
)

plt.figure(figsize=(11, 5))
sns.countplot(data=state_counts, x="state", hue="period", order=STATE_ORDER)
plt.title("Current vs Predicted Future Client States")
plt.xlabel("State")
plt.ylabel("Client-category rows")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(transitions["transition_probabilities"], annot=True, fmt=".1%", cmap="Blues")
plt.title("Forecasted State Transition Probabilities")
plt.xlabel("Future state")
plt.ylabel("Current state")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.countplot(data=alerts, x="priority", hue="future_state", order=["High", "Medium", "Low"])
plt.title("Alert Priority by Predicted Future State")
plt.tight_layout()
plt.show()

## 7. PCA: Client Behavior Map

PCA is not used for the rule-based classification itself. It is used as an exploratory visualization.

Standardization is necessary here because variables have different units: euros, ratios, counts, and weeks. Without scaling, spend variables would dominate the components.

In [ ]:
pca_features = [
    "spend_4w", "prev_spend_4w", "spend_12w", "spend_52w", "annual_potential", "potential_4w",
    "capture_ratio", "observed_trend_pct", "classification_observed_slope", "trend_4w_vs_prev_4w", "active_weeks_12w", "active_weeks_4w",
    "active_weeks_52w",
    "weekly_mean_12w", "weekly_std_12w", "volatility_12w", "weeks_since_last_purchase",
    "regularity_score", "abnormal_silence", "capture_strength", "client_weighted_historical_product_slope", "client_weighted_forecast_product_slope", "predicted_spend_4w", "predicted_capture_ratio", "predicted_capture_strength", "predicted_spend_change_pct", "forecast_trend_pct", "classification_forecast_slope",
]

pca_df = classifications[classifications["classification_eligible"]].copy()
X_pca = pca_df[pca_features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

pca = PCA(n_components=3, random_state=42)
coords = pca.fit_transform(X_scaled)
pca_plot = pca_df[["client_id", "category_h", "analytical_block", "current_state", "future_state"]].copy()
pca_plot["PC1"] = coords[:, 0]
pca_plot["PC2"] = coords[:, 1]
pca_plot["PC3"] = coords[:, 2]

display(Markdown(
    f"PCA explains **{pca.explained_variance_ratio_[0]:.1%}** on PC1 and "
    f"**{pca.explained_variance_ratio_[1]:.1%}** on PC2 and "
    f"**{pca.explained_variance_ratio_[2]:.1%}** on PC3."
))

loadings = pd.DataFrame({
    "feature": pca_features,
    "PC1_loading": pca.components_[0],
    "PC2_loading": pca.components_[1],
})
display(loadings.assign(abs_pc1=lambda d: d.PC1_loading.abs()).sort_values("abs_pc1", ascending=False).head(10))
display(loadings.assign(abs_pc2=lambda d: d.PC2_loading.abs()).sort_values("abs_pc2", ascending=False).head(10))

sample_plot = pca_plot.sample(min(8000, len(pca_plot)), random_state=42)
fig_current = px.scatter_3d(
    sample_plot,
    x="PC1",
    y="PC2",
    z="PC3",
    color="current_state",
    symbol="analytical_block",
    hover_data=["client_id", "category_h", "analytical_block", "future_state"],
    title="3D PCA Map Colored by Current State",
    opacity=0.70,
)
fig_current.update_traces(marker=dict(size=3))
current_pca_path = OUTPUT_DIR / "pca_current_state_3d.html"
fig_current.write_html(current_pca_path, include_plotlyjs="cdn")
display(IFrame(str(current_pca_path), width="100%", height=650))

fig_future = px.scatter_3d(
    sample_plot,
    x="PC1",
    y="PC2",
    z="PC3",
    color="future_state",
    symbol="analytical_block",
    hover_data=["client_id", "category_h", "analytical_block", "current_state"],
    title="3D PCA Map Colored by Forecasted Future State",
    opacity=0.70,
)
fig_future.update_traces(marker=dict(size=3))
future_pca_path = OUTPUT_DIR / "pca_future_state_3d.html"
fig_future.write_html(future_pca_path, include_plotlyjs="cdn")
display(IFrame(str(future_pca_path), width="100%", height=650))

display(Markdown(
    f"Interactive PCA files saved to `{current_pca_path}` and `{future_pca_path}`."
))

## 8. Rolling 5-Fold Time-Series Validation

We validate whether the pipeline can infer future client classifications.

The validation is chronological, not random:

1. Pick 5 historical anchor weeks.
2. At each anchor, use only data available up to that week.
3. Forecast the next 4 weeks.
4. Predict future client state.
5. Compare against the actual state computed from the true next 4 weeks.

This is stricter and more realistic than random train/test splitting for time series.

In [ ]:
def select_time_folds(weekly_sales_full, n_folds=5, min_history_weeks=60, horizon=4):
    weeks = pd.DatetimeIndex(sorted(pd.to_datetime(weekly_sales_full["week_start"].dropna().unique())))
    usable = weeks[min_history_weeks: max(min_history_weeks, len(weeks) - horizon)]
    if len(usable) < n_folds:
        raise ValueError("Not enough weekly history for the requested number of folds.")
    positions = np.linspace(0, len(usable) - 1, n_folds).round().astype(int)
    return pd.DatetimeIndex(usable[positions])


def actual_future_labels(predicted_features, weekly_sales_full, anchor_week, horizon=4):
    future_weeks_actual = pd.date_range(anchor_week + pd.Timedelta(days=7), periods=horizon, freq=sd.WEEK_FREQ)
    actual = (
        weekly_sales_full[weekly_sales_full["week_start"].isin(future_weeks_actual)]
        .groupby(["client_id", "category_h", "analytical_block"], as_index=False)["revenue"]
        .sum()
        .rename(columns={"revenue": "actual_future_spend_4w"})
    )
    out = predicted_features.merge(actual, on=["client_id", "category_h", "analytical_block"], how="left")
    out["actual_future_spend_4w"] = out["actual_future_spend_4w"].fillna(0.0).clip(lower=0.0)
    out["actual_future_capture_ratio"] = (
        out["actual_future_spend_4w"] / out["potential_4w"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0.0, upper=3.0)
    out["actual_future_trend_pct"] = sd.potential_scaled_change(out["actual_future_spend_4w"], out["spend_4w"], out["potential_4w"])
    out["actual_classification_slope"] = out["client_weighted_forecast_product_slope"].where(
        out["client_weighted_forecast_product_slope"].abs() > sd.EPS,
        out["actual_future_trend_pct"],
    )
    out["actual_future_state"] = sd.classify_state(
        out["actual_future_spend_4w"],
        out["actual_future_capture_ratio"],
        out["actual_classification_slope"],
        out["analytical_block"],
        CLASSIFICATION_CONFIG,
    )
    out = out[out["classification_eligible"]].copy()
    return out


def run_pipeline_at_anchor(data, anchor_week, forecast_weeks=4, model_mode="auto", maxiter=60):
    sales = data["sales"]
    max_week = sd.week_start(pd.Series([anchor_week])).iloc[0]
    min_week = sales["week_start"].min()
    history_weeks_local = pd.date_range(min_week, max_week, freq=sd.WEEK_FREQ)
    future_weeks_local = pd.date_range(max_week + pd.Timedelta(days=7), periods=forecast_weeks, freq=sd.WEEK_FREQ)
    all_weeks_local = history_weeks_local.union(future_weeks_local)
    campaign_calendar_local = sd.build_campaign_calendar(data["campaigns"], all_weeks_local)
    weekly_all_local = sd.build_weekly_sales(sales, campaign_calendar_local)
    weekly_history_local = weekly_all_local[weekly_all_local["week_start"].isin(history_weeks_local)].copy()
    tf, fbb = sd.forecast_type_sales(weekly_history_local, history_weeks_local, future_weeks_local, model_mode, maxiter, MODEL_CONFIG)
    pf = sd.forecast_product_sales(weekly_history_local, data["products"], history_weeks_local, future_weeks_local, fbb, model_mode, maxiter, MODEL_CONFIG)
    cf, pdv = sd.build_client_features(weekly_history_local, data["potential"], max_week, history_weeks_local, pf, CLASSIFICATION_CONFIG)
    return cf, tf, pf


def evaluate_rolling_cv(data, weekly_sales_full, folds, forecast_weeks=4, model_mode="auto", maxiter=60):
    fold_frames = []
    fold_metrics = []
    for i, anchor in enumerate(folds, start=1):
        display(Markdown(f"Running fold **{i}/{len(folds)}** with anchor week `{anchor.date()}`"))
        predicted, _, _ = run_pipeline_at_anchor(data, anchor, forecast_weeks, model_mode, maxiter)
        evaluated = actual_future_labels(predicted, weekly_sales_full, anchor, forecast_weeks)
        evaluated["fold"] = i
        evaluated["anchor_week"] = anchor
        y_true = evaluated["actual_future_state"]
        y_pred = evaluated["future_state"]
        fold_metrics.append({
            "fold": i,
            "anchor_week": anchor,
            "rows": len(evaluated),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        })
        fold_frames.append(evaluated)
    return pd.concat(fold_frames, ignore_index=True), pd.DataFrame(fold_metrics)


folds = select_time_folds(weekly_sales, n_folds=5, min_history_weeks=60, horizon=FORECAST_WEEKS)
display(pd.DataFrame({"fold": range(1, 6), "anchor_week": folds}))

cv_predictions, cv_metrics = evaluate_rolling_cv(
    data, weekly_sales, folds, FORECAST_WEEKS, CV_MODEL_MODE, MAXITER
)

display(cv_metrics)
display(Markdown(
    f"Rule-based future classification mean accuracy: **{cv_metrics['accuracy'].mean():.3f}**  \n"
    f"Rule-based mean macro-F1: **{cv_metrics['macro_f1'].mean():.3f}**"
))

STATE_LABELS = ["Loyal", "Promising", "Risky", "Promiscuous"]
print(classification_report(cv_predictions["actual_future_state"], cv_predictions["future_state"], labels=STATE_LABELS, zero_division=0))
cm = confusion_matrix(cv_predictions["actual_future_state"], cv_predictions["future_state"], labels=STATE_LABELS)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=STATE_LABELS, yticklabels=STATE_LABELS)
plt.title("Rolling CV Confusion Matrix: Rule-Based Future State")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Standardized Logistic Regression as an Improvement Candidate

The rule engine is transparent and presentation-friendly. Still, if validation is weak, a calibrated supervised model can learn better decision boundaries from historical rolling windows.

Important: the labels are generated by the same state definition applied to actual future data. This means the logistic regression is not learning human CRM labels; it is learning to approximate the rule-defined future state from current and forecasted features.

Standardization is required for logistic regression because features are on very different scales.

In [ ]:
ML_FEATURES = [
    "spend_4w", "prev_spend_4w", "spend_12w", "spend_52w", "annual_potential", "potential_4w",
    "capture_ratio", "observed_trend_pct", "classification_observed_slope", "trend_4w_vs_prev_4w", "active_weeks_12w", "active_weeks_4w",
    "active_weeks_52w",
    "weekly_mean_12w", "weekly_std_12w", "volatility_12w", "weeks_since_last_purchase",
    "regularity_score", "abnormal_silence", "capture_strength", "client_weighted_historical_product_slope", "client_weighted_forecast_product_slope", "predicted_spend_4w", "predicted_capture_ratio", "predicted_capture_strength", "predicted_spend_change_pct", "forecast_trend_pct", "classification_forecast_slope",
    "client_product_forecast_factor", "category_forecast_factor",
]


def clean_ml_matrix(df, features):
    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X


def train_ml_candidate_from_cv(cv_predictions):
    rows = cv_predictions[cv_predictions["actual_future_state"].isin(STATE_LABELS)].copy()
    label_counts = rows["actual_future_state"].value_counts()
    valid_labels = label_counts[label_counts >= 5].index
    rows = rows[rows["actual_future_state"].isin(valid_labels)].copy()
    if rows["actual_future_state"].nunique() < 2:
        return None, None, None, None

    fold_results = []
    all_preds = []
    for test_fold in sorted(rows["fold"].unique()):
        train = rows[rows["fold"] != test_fold]
        test = rows[rows["fold"] == test_fold]
        if train["actual_future_state"].nunique() < 2 or test.empty:
            continue
        scaler = StandardScaler()
        encoder = LabelEncoder()
        X_train = scaler.fit_transform(clean_ml_matrix(train, ML_FEATURES))
        y_train = encoder.fit_transform(train["actual_future_state"])
        X_test = scaler.transform(clean_ml_matrix(test, ML_FEATURES))
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=None)
        clf.fit(X_train, y_train)
        pred = encoder.inverse_transform(clf.predict(X_test))
        part = test[["fold", "client_id", "category_h", "analytical_block", "actual_future_state", "future_state"]].copy()
        part["ml_future_state"] = pred
        all_preds.append(part)
        fold_results.append({
            "fold": test_fold,
            "rows": len(test),
            "ml_accuracy": accuracy_score(test["actual_future_state"], pred),
            "ml_macro_f1": f1_score(test["actual_future_state"], pred, average="macro"),
            "ml_weighted_f1": f1_score(test["actual_future_state"], pred, average="weighted"),
        })

    if not all_preds:
        return None, None, None, None

    pred_df = pd.concat(all_preds, ignore_index=True)
    metrics = pd.DataFrame(fold_results)

    final_scaler = StandardScaler()
    final_encoder = LabelEncoder()
    X_all = final_scaler.fit_transform(clean_ml_matrix(rows, ML_FEATURES))
    y_all = final_encoder.fit_transform(rows["actual_future_state"])
    final_model = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=None)
    final_model.fit(X_all, y_all)
    return pred_df, metrics, (final_model, final_scaler, final_encoder), rows


ml_predictions, ml_metrics, ml_artifacts, ml_training_rows = train_ml_candidate_from_cv(cv_predictions)

if ml_metrics is None:
    display(Markdown("The ML improvement candidate could not be trained because there were not enough historical classes."))
else:
    display(ml_metrics)
    rule_macro = cv_metrics["macro_f1"].mean()
    ml_macro = ml_metrics["ml_macro_f1"].mean()
    champion = "ML logistic regression" if ml_macro > rule_macro else "transparent rule engine"
    display(Markdown(
        f"Rule mean macro-F1: **{rule_macro:.3f}**  \n"
        f"ML mean macro-F1: **{ml_macro:.3f}**  \n"
        f"Recommended champion for the demo: **{champion}**."
    ))
    print(classification_report(ml_predictions["actual_future_state"], ml_predictions["ml_future_state"], labels=STATE_LABELS, zero_division=0))

    cm_ml = confusion_matrix(ml_predictions["actual_future_state"], ml_predictions["ml_future_state"], labels=STATE_LABELS)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm_ml, annot=True, fmt="d", cmap="Greens", xticklabels=STATE_LABELS, yticklabels=STATE_LABELS)
    plt.title("Rolling CV Confusion Matrix: Logistic Regression Candidate")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

## 10. Logistic Regression Interpretation

This section explains which standardized features push the ML candidate toward each state. Coefficients are only shown if the logistic regression candidate was trained successfully.

In [ ]:
if ml_artifacts is not None:
    model, scaler, encoder = ml_artifacts
    coef = pd.DataFrame(model.coef_, columns=ML_FEATURES, index=encoder.classes_).T
    for cls in encoder.classes_:
        display(Markdown(f"### Strongest standardized coefficients for `{cls}`"))
        display(coef[[cls]].assign(abs_coef=lambda d: d[cls].abs()).sort_values("abs_coef", ascending=False).head(10))
else:
    display(Markdown("No ML coefficients available."))

## 11. Alert Interpretation and Export

Each alert includes:

- Current state and predicted future state.
- Product/category context.
- Priority score.
- Predicted spend change.
- Potential gap.
- Main product driver.
- Human-readable explanation.

This is the main operational output for the commercial team.

In [ ]:
display(alerts[[
    "client_id", "category_h", "analytical_block", "current_state", "future_state",
    "priority", "priority_score", "spend_4w", "predicted_spend_4w",
    "classification_forecast_slope", "potential_gap_4w", "driver_product_id",
    "driver_forecast_product_slope_pct", "explanation"
]].head(30))

sd.write_outputs(
    OUTPUT_DIR,
    weekly_sales,
    type_forecasts,
    product_forecasts,
    classifications,
    alerts,
    {
        "input": str(INPUT_FILE),
        "current_week": current_week,
        "forecast_weeks": FORECAST_WEEKS,
        "model_mode": MODEL_MODE,
        "cv_model_mode": CV_MODEL_MODE,
        "cv_rule_accuracy_mean": float(cv_metrics["accuracy"].mean()),
        "cv_rule_macro_f1_mean": float(cv_metrics["macro_f1"].mean()),
        "cv_ml_macro_f1_mean": None if ml_metrics is None else float(ml_metrics["ml_macro_f1"].mean()),
        "alerts": int(len(alerts)),
        "high_priority_alerts": int((alerts.get("priority") == "High").sum()) if not alerts.empty else 0,
        "classification_config": CLASSIFICATION_CONFIG,
        "model_config": MODEL_CONFIG,
    },
    TOP_ALERTS,
    transitions,
)

for name, table in transitions.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv")

cv_predictions.to_csv(OUTPUT_DIR / "rolling_cv_rule_predictions.csv", index=False)
cv_metrics.to_csv(OUTPUT_DIR / "rolling_cv_rule_metrics.csv", index=False)
if ml_metrics is not None:
    ml_predictions.to_csv(OUTPUT_DIR / "rolling_cv_ml_predictions.csv", index=False)
    ml_metrics.to_csv(OUTPUT_DIR / "rolling_cv_ml_metrics.csv", index=False)

display(Markdown(f"Outputs written to `{OUTPUT_DIR}`."))

## 12. Final Interpretation Checklist

Use these points when presenting the solution:

- The system does not claim to observe competitor purchases directly; it infers commercial risk from potential achievement and forecasted demand change.
- Commodities and technical products are treated differently through configurable block-specific thresholds.
- SARIMA/SARIMAX forecasts are used as demand signals and explanation drivers.
- The final four states are exactly: `Loyal`, `Promiscuous`, `Risky`, and `Promising`.
- Transition matrices estimate expected movement between states over the next 4 weeks.
- PCA helps visualize whether states occupy meaningfully different behavioral regions.
- Rolling 5-fold validation tests whether predicted future states match states observed from actual future weeks.
- If logistic regression beats the rules, use it as a second-stage scorer; if not, keep the transparent rules for the demo.